In [44]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_ROW_HEIGHT
from datetime import datetime
import pandas as pd
import os
from docx.oxml.shared import OxmlElement, qn
from docx.oxml.ns import nsdecls
from docx.opc.constants import RELATIONSHIP_TYPE
from datetime import datetime
import io
from PIL import Image
# PATHS SETUP

monthly_metrics_path = 'data/monthly_metrics.csv'
observations_path = 'data/observations.csv'
new_species_path = 'data/new_species_photo.csv'
taxon_counts_path = 'data/taxon_counts.csv'
main_metrics_path = 'data/main_metrics.csv'

new_species_figures_path = 'figures/photos_new_species'

# DATA LOADING

df_monthly = pd.read_csv(monthly_metrics_path)
df_new_species = pd.read_csv(new_species_path)
df_taxon_count = pd.read_csv(taxon_counts_path)
df_observations = pd.read_csv(observations_path)
df_main_metrics = pd.read_csv(main_metrics_path)



# COLOR PALETTES

hex_color_orange = 'f85532'
hex_color_blue = '2b2e4f'

palette_orange = RGBColor.from_string(hex_color_orange)
palette_blue = RGBColor.from_string(hex_color_blue)


In [45]:
def add_hyperlink(paragraph, url, text):
    
    """Añade un hipervínculo real que funciona en Word"""
    part = paragraph.part
    r_id = part.relate_to(url, RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    
    hyperlink = OxmlElement('w:hyperlink')
    hyperlink.set(qn('r:id'), r_id)
    
    new_run = OxmlElement('w:r')
    rPr = OxmlElement('w:rPr')
    
    # Estilo azul subrayado
    color = OxmlElement('w:color')
    color.set(qn('w:val'), "0000FF")
    rPr.append(color)
    
    underline = OxmlElement('w:u')
    underline.set(qn('w:val'), 'single')
    rPr.append(underline)
    
    new_run.append(rPr)
    t = OxmlElement('w:t')
    t.text = text
    new_run.append(t)
    
    hyperlink.append(new_run)
    paragraph._p.append(hyperlink)


In [46]:
def get_minka_docx():
    
    last_month = datetime.now().month - 1
    current_year = datetime.now().year

    doc = Document()
      
    doc.add_paragraph()  

    style = doc.styles['Normal']
    font = style.font
    font.name = 'Times New Roman'
    font.size = Pt(12)
    doc.styles['Normal'].language_id = 1027

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title.add_run("INFORME MENSUAL DEL PROJECTE")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(24)
    run.font.color.rgb = palette_orange

    
    fecha_paragraph = doc.add_paragraph()  

    run_fecha = fecha_paragraph.add_run(F"Informe del {last_month:02d} de {current_year}")  
    run_fecha.font.name = 'Arial Rounded MT Bold' 
    run_fecha.font.size = Pt(18)
    run_fecha.font.color.rgb = palette_blue
    fecha_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_page_break()

    intro = ("En aquest informe es presenten les principals mètriques i estadístiques del projecte. "
        "Aquest document ha estat elaborat a partir de dades de ciència ciutadana "
        "recollides a través de la plataforma web de Minka, una eina col·laborativa per a l’observació "
        "i registre de la biodiversitat.\n"
        "L’objectiu d’aquest informe és oferir una visió general dels resultats obtinguts i facilitar "
        "l’anàlisi de la participació i de les dades observacionals registrades pels usuaris."
    )

    doc.add_paragraph(intro)

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Principals mètriques observades l'últim mes")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue


    intro_main_metrics = ("Les dades que es presenten a continuació corresponen a un resum mensual generat de "
    "manera automàtica a partir dels registres disponibles a la plataforma Minka. Aquestes mètriques reflecteixen "
    "l’estat actual del projecte i la seva evolució en els darrers 30 dies pels usuaris."
    )

    doc.add_paragraph(intro_main_metrics)

    table = doc.add_table(rows=1, cols=3)
    table.style = 'Table Grid'

    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'Mètrica'
    hdr_cells[1].text = 'Valor actual'
    hdr_cells[2].text = 'Variació últim mes'

    for cell in hdr_cells:
        run = cell.paragraphs[0].runs[0]
        run.bold = True
        cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER 
    title_metrics = {
        "observations": "Observacions",
        "observers": "Observadors",
        "identifiers": "Identificadors",
        "species": "Espècies"
    }

    for _, row in df_main_metrics.iterrows():
        row_cells = table.add_row().cells


        metric_ca = title_metrics.get(row['metric'].lower(), row['metric'].capitalize())
        para0 = row_cells[0].paragraphs[0]
        para0.text = metric_ca
        para0.alignment = WD_ALIGN_PARAGRAPH.LEFT

        para1 = row_cells[1].paragraphs[0]
        para1.text = str(row['number_today'])
        para1.alignment = WD_ALIGN_PARAGRAPH.RIGHT

        para2 = row_cells[2].paragraphs[0]
        para2.text = str(row['number_in_last_month'])
        para2.alignment = WD_ALIGN_PARAGRAPH.RIGHT

    
    doc.add_page_break()
    
    
    # PLOTS MONTHLY METRICS
    
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Representació gràfica de les métriques principals de forma mensual")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue
    
    intro_grafics = "A continuació, es presenten els gràfics corresponents a les principals variables analitzades: "
    "Observacions, observadors, identificadors i espècies. Aquests gràfics tenen com a objectiu proporcionar "
    "una visió general  de l'activitat registrada al vostre projecte.\n\n"

    doc.add_paragraph(intro_grafics)
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Observacions")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue

    doc.add_picture("figures/monthly_metrics/monthly_observations.png", width=Inches(5.5))

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Observadors")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue

    doc.add_picture("figures/monthly_metrics/monthly_observers.png", width=Inches(5.5))
    
    doc.add_page_break()

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Identificadors")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue

    doc.add_picture("figures/monthly_metrics/monthly_identifiers.png", width=Inches(5.5))

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Espècies")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue

    doc.add_picture("figures/monthly_metrics/monthly_species.png", width=Inches(5.5))
    
    doc.add_page_break()


    # TAXON PLOTS

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Seguiment de les taxonomies a diferents nivellls")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue
    
    intro_taxon_plots = 'A continuació, es presenten els gràfics corresponents als diferents nivells taxonomics:' \
    'Regne, familia, clase, filo i espècie'

    doc.add_paragraph(intro_taxon_plots)

    
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Classificació per regne")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue
   
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/taxon_plots/top_kingdom.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_page_break()


    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Classificació per clase")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue
   
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/taxon_plots/top_class.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Classificació per filo")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue
   
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/taxon_plots/top_phylum.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER


    doc.add_page_break()

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Classificació per família")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue

    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/taxon_plots/top_family.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Classificació per espècies")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue
   
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/taxon_plots/top_species.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER


    doc.add_page_break()

    # HEATMAP PLOTS
    
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Mapes de calor")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue

    intro_heatmap = ("Finalment,en aquest apartat es presenta un mapa de calor que mostra la distribució espacial de les observacions registrades. "
        "La densitat d’observacions es representa mitjançant una escala de colors que facilita la detecció de patrons espacials, sent de color vermell les zones amb alta densitat d'observacions" \
        "i les blaves zones amb baixa densitat d'observacions \n\n"
    )

    doc.add_paragraph(intro_heatmap)

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Mapa de calor de les observacions totals")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue

    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/heatmap_plots/heatmap_image_None.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    
    doc.add_page_break()
    
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Mapa de calor de la clase animalia")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue
    
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/heatmap_plots/heatmap_image_animalia.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Mapa de calor de la clase aves")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue
    
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/heatmap_plots/heatmap_image_aves.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    
    doc.add_page_break()
    
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Mapa de calor de la clase plantae")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue
    
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/heatmap_plots/heatmap_image_plantae.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Mapa de calor de la clase fungi")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(12)
    run.font.color.rgb = palette_blue
     
    paragraph_heatmap = doc.add_paragraph()
    run = paragraph_heatmap.add_run()
    run.add_picture("figures/heatmap_plots/heatmap_image_fungi.png", width=Inches(6))
    paragraph_heatmap.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_page_break()

    doc.add_page_break()
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = title.add_run("Noves espècies registrades")
    run.font.name = 'Arial Rounded MT Bold'  
    run.font.size = Pt(14)
    run.font.color.rgb = palette_blue

    intro_new_species = (
        "A continuació es presenten totes les espècies novament registrades a la plataforma "
        "durant el període considerat. Feu clic a 'Enllaç' per veure l'observació original."
    )
    doc.add_paragraph(intro_new_species)

  
    table = doc.add_table(rows=1, cols=4)
    table.style = 'Table Grid'

    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = "Nom de l'espècie"
    hdr_cells[1].text = "Observat al"
    hdr_cells[2].text = "Usuari"
    hdr_cells[3].text = "Enllaç"

    for cell in hdr_cells:
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.font.bold = True
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)

    for _, row in df_new_species.iterrows():
        row_cells = table.add_row().cells
        row_cells[0].text = str(row['taxon_name'])
        row_cells[1].text = str(row['observed_on'])
        row_cells[2].text = str(row['user_login'])
        
        paragraph = row_cells[3].paragraphs[0]
        add_hyperlink(paragraph, row['obs_url'], "Enllaç")



    photo_files = [f for f in os.listdir(new_species_figures_path) if f.lower().endswith(('.jpeg'))]
    photo_files = photo_files[:5]

    if photo_files:
        doc.add_page_break()
        title = doc.add_paragraph()
        title.alignment = WD_ALIGN_PARAGRAPH.LEFT
        run = title.add_run("Algunes fotografies de les noves espècies registrades")
        run.font.name = 'Arial Rounded MT Bold'
        run.font.size = Pt(14)
        run.font.color.rgb = palette_blue

        for photo_name in photo_files:
            image_path = os.path.join(new_species_figures_path, photo_name)
            
            with Image.open(image_path) as img:
                img = img.convert("RGB")  
                img.thumbnail((800, 800))  
                temp_path = os.path.join(new_species_figures_path, f"resized_{photo_name}")
                img.save(temp_path, format="JPEG", quality=60) 

            paragraph_img = doc.add_paragraph()
            run_img = paragraph_img.add_run()
            run_img.add_picture(temp_path, width=Inches(5.5))
            paragraph_img.alignment = WD_ALIGN_PARAGRAPH.CENTER

    paragraf_final = doc.add_paragraph()
    paragraf_final.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run_final = paragraf_final.add_run(
    "La resta de fotografies de les noves espècies registrades es poden consultar a la carpeta: figures/photos_new_species"
    )
    run_final.font.name = 'Times New Roman'
    run_final.font.size = Pt(12) 

    doc.save("informe_mensual_minka.docx")


In [47]:
get_minka_docx()